# COMP 3404 - Final Project - Interactive Dashboard 

### By: Hageer Birama - 202034484 & Nicole Bishop-Adigwe - 201960085
### Due: Sunday April 12, 2026

#### Import dependencies

In [1]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from scipy.stats import gaussian_kde as gkde
import json, urllib.request, os
import ipywidgets as widgets
from IPython.display import display, clear_output
#import warnings

In [2]:
# handle ssl issues when downloading map from external web
import ssl
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

#### Load data and preprocess it

In [3]:
# load dataset
df = pd.read_csv('birds.csv')

# remove extra spaces from column names
df.columns = df.columns.str.strip()

In [4]:
# format observation date into datetime 
df['OBSERVATION DATE'] = pd.to_datetime(df['OBSERVATION DATE'], errors='coerce')
df = df.dropna(subset=['OBSERVATION DATE'])     # remove rows where date it missing

# extract month number and month name for grouping
df['month_num']  = df['OBSERVATION DATE'].dt.month
df['month_name'] = df['OBSERVATION DATE'].dt.strftime('%b')

# convert the Xs to 1s
df['OBSERVATION COUNT'] = pd.to_numeric(
    df['OBSERVATION COUNT'].replace('X', 1), errors='coerce'
).fillna(1)

In [5]:
# remove rows without LAT or LONG
df = df.dropna(subset=['LATITUDE', 'LONGITUDE'])

# only want NL data
df = df[
    df['LATITUDE'].between(46.0, 61.0) &
    df['LONGITUDE'].between(-68.0, -52.0)
]

# order of months
MONTH_ORDER = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']

In [6]:
# list of counties for dropfdown widget
ALL_COUNTIES = ['All regions'] + sorted(df['COUNTY'].dropna().unique().tolist())

# top species for default selctions of plot
TOP_SPECIES_GLOBAL = (
    df.groupby('COMMON NAME')['OBSERVATION COUNT']
    .sum().nlargest(30).index.tolist()
)

#### Load NL map via GeoJSON file

In [7]:
NL_GEOJSON_URL = (
    'https://raw.githubusercontent.com/codeforgermany/click_that_hood/'
    'main/public/data/canada.geojson'
)
GEOJSON_CACHE = 'canada_provinces.geojson'

nl_polygons = []
other_polys = []

try:
    # download file if not already downloaded
    if not os.path.exists(GEOJSON_CACHE):
        urllib.request.urlretrieve(NL_GEOJSON_URL, GEOJSON_CACHE)
        
    with open(GEOJSON_CACHE) as fh:
        gj = json.load(fh)

    # extracts ploygon coordinates 
    def extract_polys(feat):
        polys, geom = [], feat['geometry']

        # for both signle and multipolys
        if geom['type'] == 'Polygon':
            c = geom['coordinates'][0]
            polys.append(([x[0] for x in c], [x[1] for x in c]))
            
        elif geom['type'] == 'MultiPolygon':
            for poly in geom['coordinates']:
                c = poly[0]
                polys.append(([x[0] for x in c], [x[1] for x in c]))
                
        return polys

    # isolate NL from other provinces
    for feat in gj['features']:
        name = feat['properties'].get('name', '')
        if 'Newfoundland' in name or 'Labrador' in name:
            nl_polygons.extend(extract_polys(feat))
        elif name in ('Quebec', 'Nova Scotia', 'New Brunswick',
                      'Prince Edward Island'):
            other_polys.extend(extract_polys(feat))
            
    print(f'NL polygons: {len(nl_polygons)}')
    
except Exception as e:
    print(f'GeoJSON failed: {e}')

NL polygons: 21


In [8]:
# draws the map of NL
def draw_nl_map(ax):
    for lons, lats in other_polys:
        ax.fill(lons, lats, color='#c8d8c0', alpha=0.55, zorder=1)
        ax.plot(lons, lats, color='#99aaa0', lw=0.4, alpha=0.7, zorder=2)
    for lons, lats in nl_polygons:
        ax.fill(lons, lats, color='#b8d4a0', alpha=0.85, zorder=3)
        ax.plot(lons, lats, color='#6a9a72', lw=0.6, alpha=0.90, zorder=4)

#### Colour theme for the dashboard 

In [9]:
# colours for background, accents, titles, texts, map (ocean & land)
BG_DARK  = '#0D1B2A'
BG_PANEL = '#112233'
ACCENT1  = '#F4A261'
ACCENT2  = '#2A9D8F'
ACCENT3  = '#E9C46A'
ACCENT4  = '#E76F51'
TEXT_MAIN = '#F0EAD6'
TEXT_DIM = '#8899AA'
GRID_CLR = '#1E3048'
OCEAN_CLR = '#a8d4e6'

In [10]:
# gradient for map 
geo_cmap = LinearSegmentedColormap.from_list(
    'geo', ['#1a0050','#8B008B','#FF8C00','#FFD700'], N=256)

In [11]:
# styling of dashboard 
plt.rcParams.update({
    'figure.facecolor': BG_DARK,
    'axes.facecolor'  : BG_PANEL,
    'text.color'      : TEXT_MAIN,
    'axes.labelcolor' : TEXT_DIM,
    'xtick.color'     : TEXT_DIM,
    'ytick.color'     : TEXT_DIM,
    'axes.edgecolor'  : GRID_CLR,
    'grid.color'      : GRID_CLR,
    'legend.facecolor': BG_DARK,
    'legend.edgecolor': GRID_CLR,
    'axes.titlecolor' : ACCENT3,
    'axes.titlesize'  : 14,
    'axes.labelsize'  : 11,
})

SPECIES_COLORS = [
    ACCENT1, ACCENT2, ACCENT3, ACCENT4,
    '#48CAE4','#90E0EF','#ADE8F4',
    '#C77DFF','#9B5DE5','#F15BB5'
]
print('Theme set.')

Theme set.


#### Visulaizations & Widgets

In [12]:
# VISULAIZATION 1 - SEASONAL TRENDS (LINE CHART)

# users select species they want to see to compare trends across months
w_species = widgets.SelectMultiple(
    options=TOP_SPECIES_GLOBAL, value=TOP_SPECIES_GLOBAL[:5],
    description='Species:',
    layout=widgets.Layout(height='180px', width='340px'),
    style={'description_width': '70px'})

# users filter the months they want to see
w_months = widgets.IntRangeSlider(
    value=[1, 12], min=1, max=12, description='Months:',
    continuous_update=False,
    style={'description_width': '70px'},
    layout=widgets.Layout(width='400px'))

# label of widget
month_label = widgets.Label(value='Jan – Dec')

# update the months seen 
def _upd_month(c):
    lo, hi = w_months.value
    month_label.value = f'{MONTH_ORDER[lo-1]} – {MONTH_ORDER[hi-1]}'
    
w_months.observe(_upd_month, names='value')

out1 = widgets.Output()


# visulization
def plot_viz1(change=None):

    # get species and months from widget input
    selected = list(w_species.value)
    m_lo, m_hi = w_months.value

    with out1:
        clear_output(wait=True)

        # don't allow empty plot w/ no selections
        if not selected:
            print('Please select at least one species.')
            return

        # filter dataset based on selections
        sub = df[
            df['COMMON NAME'].isin(selected) &
            df['month_num'].between(m_lo, m_hi)]

        # aggregate total observations per species per month & pivot so each species becomes it own coloumn (line plotting)
        pivot = (
            sub.groupby(['COMMON NAME','month_num'])['OBSERVATION COUNT'].sum()
            .reset_index()
            .pivot(index='month_num', columns='COMMON NAME',
                   values='OBSERVATION COUNT')
            .reindex(range(m_lo, m_hi+1)).fillna(0))

        # create figure
        fig, ax = plt.subplots(figsize=(13, 5), facecolor=BG_DARK)
        ax.set_facecolor(BG_PANEL)

        # plot each species as spearate line
        for i, sp in enumerate(pivot.columns):
            c = SPECIES_COLORS[i % len(SPECIES_COLORS)]

            # line for species showing trend
            ax.plot(pivot.index, pivot[sp], marker='o', markersize=5,
                    linewidth=2.2, color=c, label=sp)

            # filling in line below (think it looks good?)
            ax.fill_between(pivot.index, pivot[sp], alpha=0.08, color=c)

        # plot title and styling
        ax.set_xticks(range(m_lo, m_hi+1))
        ax.set_xticklabels([MONTH_ORDER[m-1] for m in range(m_lo, m_hi+1)])
        ax.set_xlabel('Month')
        ax.set_ylabel('Total birds observed')
        
        ax.set_title('1. Seasonal Trends — Monthly Observations per Species',
                     fontsize=14, color=ACCENT3, fontweight='bold')
        
        ax.yaxis.set_major_formatter(
            ticker.FuncFormatter(lambda x,_: f'{x/1000:.0f}k' if x>=1000 else str(int(x))))
        
        ax.grid(True, axis='y', alpha=0.4)       # put grid on plot
        
        ax.legend(loc='upper right', fontsize=8.5,
                  framealpha=0.4, labelcolor=TEXT_MAIN)
        
        plt.tight_layout()
        plt.show()

        plt.close(fig)     # so it can update after widget inputs
        
w_species.observe(plot_viz1, names='value')
w_months.observe(plot_viz1, names='value')



# VISULIZATION 2 - BIRDWATCHING HOTSPOTS IN NL (GEOGRAPHICAL SCATTER)

loc_all = (
    df.groupby(['COUNTY','LOCALITY','LATITUDE','LONGITUDE'])
    .agg(total_obs=('OBSERVATION COUNT','sum'),
         num_checklist=('GLOBAL UNIQUE IDENTIFIER','count'))
    .reset_index())

# users filter by county w/ dropdown selection
w_county = widgets.Dropdown(
    options=ALL_COUNTIES, value='All regions', description='Region:',
    style={'description_width':'70px'},
    layout=widgets.Layout(width='320px'))

# users set min count of observations to explore hotspots in NL based on county
max_obs = int(loc_all['total_obs'].quantile(0.90))
w_min_obs = widgets.IntSlider(
    value=0, min=0, max=max_obs,
    step=max(1, max_obs//50),
    description='Min obs:', continuous_update=False,
    style={'description_width':'70px'},
    layout=widgets.Layout(width='380px'))

# users zoom into region selected by dropdown
w_zoom = widgets.ToggleButton(
    value=False,
    description='Zoom to region',
    button_style='info',    # blue tint when active
    tooltip='Toggle to zoom the map into the currently filtered region',
    icon='search-plus',
    layout=widgets.Layout(width='160px')
)

out2 = widgets.Output()

# visulization
def plot_viz2(change=None):

    # get slected inputs from widgets
    county  = w_county.value
    min_obs = w_min_obs.value
    zoom_on = w_zoom.value
    
    with out2:
        clear_output(wait=True)

        # begin with full dataset
        sub = loc_all.copy()

        # apply region filter if selected
        if county != 'All regions':
            sub = sub[sub['COUNTY'] == county]

        # apply county filter if selected 
        sub = sub[sub['total_obs'] >= min_obs]

        # handle case when no data matches filters
        if sub.empty:
            print('No localities match these filters.'); return

        # create figure
        fig, ax = plt.subplots(figsize=(13, 8), facecolor=BG_DARK)
        ax.set_facecolor(OCEAN_CLR)

        # draw NL 
        draw_nl_map(ax)

        # scale points based on number of observations
        sizes  = np.clip(np.sqrt(sub['num_checklist']) * 7, 8, 600)

        # colour points based off total observations & scale them
        colors = np.log1p(sub['total_obs'])

        # plot the scattered points
        sc = ax.scatter(
            sub['LONGITUDE'], sub['LATITUDE'],
            c=colors, s=sizes, cmap=geo_cmap,
            alpha=0.82, linewidths=0.35,
            edgecolors='#ffffff60', zorder=5)

        # label top 5 in filtered selection
        top5 = sub.nlargest(5, 'num_checklist').reset_index(drop=True)
        OFFSETS = [(0.7,0.6),(-2.8,0.9),(0.6,-0.9),(-2.2,-1.0),(1.1,-0.6)]
        
        for i, row in top5.iterrows():
            lbl = row['LOCALITY'].split('--')[-1].strip()[:30]
            
            dx, dy = OFFSETS[i % len(OFFSETS)]
            
            ax.annotate(
                lbl,
                xy=(row['LONGITUDE'], row['LATITUDE']),
                xytext=(row['LONGITUDE']+dx, row['LATITUDE']+dy),
                fontsize=8.5, color='#1a1a1a', fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='#333333', lw=1),
                bbox=dict(boxstyle='round,pad=0.22',
                          fc='#ffffffcc', ec='#6a9a72', lw=0.8),
                zorder=8)

        # side gradient for colour scale of points
        cbar = fig.colorbar(sc, ax=ax, orientation='vertical',
                            pad=0.01, fraction=0.018)

        # styling of colour bar
        cbar.set_label('Total obs. (log)', color=TEXT_DIM, fontsize=10)
        cbar.ax.yaxis.set_tick_params(labelcolor=TEXT_DIM, labelsize=8)
        cbar.outline.set_edgecolor('#999999')

        
        for n, lbl in [(3,'3'),(25,'25'),(100,'100+')]:
            ax.scatter([],[],s=np.clip(np.sqrt(n)*7,8,600),
                       c='#888888',alpha=0.65,label=f'{lbl} checklists')

        # legend for size of points
        ax.legend(title='Checklists', title_fontsize=9, fontsize=9,
                  loc='lower left', framealpha=0.80,
                  facecolor='#ffffffdd', edgecolor='#999999',
                  labelcolor='#222222')
        
        # use full size of NL as default
        geo_full = df

        full_lon_min = geo_full['LONGITUDE'].min() - 1.0
        full_lon_max = geo_full['LONGITUDE'].max() + 1.0
        full_lat_min = geo_full['LATITUDE'].min()  - 0.8
        full_lat_max = geo_full['LATITUDE'].max()  + 0.8

        # if zoom selected
        if zoom_on and county != 'All regions' and not sub.empty:
            # zoom into region w/ padding to include whole section,
            PAD_LON = 0.5
            PAD_LAT = 0.4
            
            ax.set_xlim(sub['LONGITUDE'].min() - PAD_LON,
                        sub['LONGITUDE'].max() + PAD_LON)
            
            ax.set_ylim(sub['LATITUDE'].min()  - PAD_LAT,
                        sub['LATITUDE'].max()  + PAD_LAT)
        else:
            # show full NL
            ax.set_xlim(full_lon_min, full_lon_max)
            ax.set_ylim(full_lat_min, full_lat_max)

        # plot title and styling
        ax.set_xlabel('Longitude', color='#444')
        ax.set_ylabel('Latitude',  color='#444')
        
        ax.tick_params(colors='#555', labelsize=9)
        ax.grid(color='#cccccc', lw=0.35, alpha=0.6, ls='--')

        
        for sp in ax.spines.values():
            sp.set_edgecolor('#999')

        # default plot
        region = county if county != 'All regions' else 'Newfoundland & Labrador'

        # updates title based on selections
        ax.set_title(
            f'2. Birdwatching Hotspots — {region}  '
            f'({len(sub):,} localities, min {min_obs:,} obs)',
            fontsize=13, color=ACCENT3, fontweight='bold')
        
        plt.tight_layout()
        plt.show()

        plt.close(fig)

w_county.observe(plot_viz2, names='value')
w_min_obs.observe(plot_viz2, names='value')
w_zoom.observe(plot_viz2, names='value')


# VISUALIZATION 3 - SPECIES OBSERVATION DISTRIBUTION (VIOLIN W/ MEAN, MEDIAN)

# find the statistics for each species
species_stats = (
    df.groupby('COMMON NAME')['OBSERVATION COUNT']
    .agg(total='sum', median='median', mean='mean')
    .reset_index()
    .sort_values('total', ascending=False)
)

# Pre-sample violin data for top 30 species
violin_pre = (
    df[df['COMMON NAME'].isin(species_stats.head(30)['COMMON NAME'])]
    .groupby('COMMON NAME')
    .apply(lambda g: g.sample(min(len(g), 400), random_state=42))
    .reset_index(drop=True)
)

# users can choose how many top species to display in plot
w_topn = widgets.IntSlider(
    value=15, min=5, max=30, step=1,
    description='Top N:', continuous_update=False,
    style={'description_width':'65px'},
    layout=widgets.Layout(width='340px'))

# users can sort plot based on mean, median, or total observations
w_sortby = widgets.Dropdown(
    options=[('Sort by total observations','total'),
             ('Sort by median count','median'),
             ('Sort by mean count','mean')],
    value='total', description='Sort by:',
    style={'description_width':'70px'},
    layout=widgets.Layout(width='300px'))

out3 = widgets.Output()

# visualization 
def plot_viz3(change=None):

    # get slected inputs from widgets
    top_n  = w_topn.value
    sortby = w_sortby.value
    
    with out3:
        clear_output(wait=True)

        # select top bird species and sort based on inputs of widget
        top_sp = (
            species_stats.head(top_n)
            .sort_values(sortby, ascending=True)['COMMON NAME'].tolist()
        )

        # map species to positions
        sp_to_y = {sp: i for i, sp in enumerate(top_sp)}
        n_sp = len(top_sp)     # number fo plots to display 

        # violin plot colours
        viol_cmap = LinearSegmentedColormap.from_list(
            'vc', ['#2A9D8F','#E76F51'], N = n_sp)

        # colours for plots 
        viol_colors = [viol_cmap(i/n_sp) for i in range(n_sp)]

        fig_h = max(6, n_sp * 0.52)     # what

        # create figure
        fig, ax = plt.subplots(figsize=(13, fig_h), facecolor=BG_DARK)
        ax.set_facecolor(BG_PANEL)

        viol_data = [violin_pre[violin_pre['COMMON NAME']==sp]
                     ['OBSERVATION COUNT'].values for sp in top_sp]
        
        MAX_X = np.percentile(np.concatenate(viol_data), 99) if viol_data else 100

        # draw plot use Kernel Density Estimation (KDE)
        for i, (sp, data) in enumerate(zip(top_sp, viol_data)):
            if len(data) < 3:
                continue
                
            y_pos = sp_to_y[sp]
            color = viol_colors[i]
            x_clip = data[data <= MAX_X]
            
            if len(x_clip) < 3:
                continue
                
            try:
                # estimate desnity 
                kde_fn  = gkde(x_clip, bw_method='scott')
                x_eval  = np.linspace(0, MAX_X, 300)
                density = kde_fn(x_eval)

                # normalize the width
                density = density / density.max() * 0.40

                # make plots symmetrical
                ax.fill_betweenx(x_eval, y_pos-density, y_pos+density,
                                 color=color, alpha=0.55, zorder=2)

                # plot both sides
                ax.plot(y_pos+density, x_eval, color=color, lw=0.7, alpha=0.8, zorder=3)
                ax.plot(y_pos-density, x_eval, color=color, lw=0.7, alpha=0.8, zorder=3)
                
            except Exception:
                pass

        # include mean and median markers on plot
        for sp in top_sp:
            y_pos  = sp_to_y[sp]
            data   = violin_pre[violin_pre['COMMON NAME']==sp]['OBSERVATION COUNT'].values
            
            if len(data) == 0:
                continue
                
            med = np.median(data)
            mn = np.mean(data)

            # plot ?? 
            ax.plot([y_pos, y_pos], [0, med],
                    color='#E9C46A', lw=1.4, alpha=0.7, zorder=4)

            # plot median 
            ax.scatter(y_pos, med, s=50, color='#E9C46A', zorder=5,
                       edgecolors=BG_DARK, linewidths=0.8)

            # plot mean 
            ax.scatter(y_pos, mn,  s=28, color='#F4A261', marker='D',
                       zorder=5, edgecolors=BG_DARK, linewidths=0.6)

        # plot title and styling 
        ax.set_xticks(range(n_sp))
        ax.set_xticklabels(top_sp, rotation=35, ha='right',
                           rotation_mode='anchor', fontsize=9, color=TEXT_MAIN)
        
        ax.set_ylabel('Observation count per checklist', fontsize=10, color=TEXT_DIM)
        
        ax.set_xlim(-0.6, n_sp-0.4)
        ax.set_ylim(-2, MAX_X*1.05)
        
        ax.yaxis.set_major_formatter(
            ticker.FuncFormatter(
                lambda x,_: f'{x/1000:.0f}k' if x>=1000 else str(int(max(x,0)))))
        
        ax.tick_params(axis='x', length=0)
        ax.grid(axis='y', color=GRID_CLR, lw=0.5, alpha=0.6)
        
        ax.set_axisbelow(True)
        for sp in ax.spines.values(): 
            sp.set_edgecolor(GRID_CLR)

        # legend for pliot -> distinguished b/w what is viloin, median, and mean
        legend_els = [
            mpatches.Patch(facecolor='#2A9D8F', alpha=0.6,
                           label='Distribution (violin)'),
            Line2D([0],[0], marker='o', color='none',
                   markerfacecolor='#E9C46A', markersize=8, label='Median'),
            Line2D([0],[0], marker='D', color='none',
                   markerfacecolor='#F4A261', markersize=6, label='Mean'),
        ]
        
        ax.legend(handles=legend_els, loc='upper right',
                  fontsize=9, framealpha=0.3,
                  facecolor=BG_DARK, labelcolor=TEXT_MAIN)
        
        sort_labels = {'total':'Total Observations',
                       'median':'Median Count', 'mean':'Mean Count'}

        # title updates 
        ax.set_title(
            f'3. Top {top_n} Species — Distribution of Sighting Counts '
            f'(sorted by {sort_labels[sortby]})',
            fontsize=13, color=ACCENT3, fontweight='bold')
        
        plt.tight_layout()
        plt.show()

        plt.close(fig)

w_topn.observe(plot_viz3, names='value')
w_sortby.observe(plot_viz3, names='value')


#### Wrap the elemnts so everything displays & in the correct order 

In [13]:
# VISUALIZATION 1
visual1_container = widgets.VBox([
    widgets.HTML('<h3 style="color:#E9C46A;font-family:Georgia,serif;'
                 'margin:16px 0 4px">1) Seasonal Trends</h3>'
                 '<p style="color:#8899AA;margin:0 0 8px;font-size:.9em">'
                 'How do observations change across the year for selected species?</p>'),
    widgets.HBox([
        widgets.VBox([widgets.Label('Select species (Ctrl/Cmd+click):'), w_species]),
        widgets.VBox([widgets.Label('Filter months:'), w_months, month_label])]),
    out1
])

# VISUALIZATION 2
visual2_container = widgets.VBox([
    widgets.HTML('<h3 style="color:#E9C46A;font-family:Georgia,serif;'
                 'margin:24px 0 4px">2) Birdwatching Hotspots</h3>'
                 '<p style="color:#8899AA;margin:0 0 8px;font-size:.9em">'
                 'Where are birds observed across Newfoundland & Labrador?</p>'),
    widgets.HBox([
        widgets.VBox([widgets.Label('Filter by region:'), w_county]),
        widgets.VBox([widgets.Label('Minimum total observations:'), w_min_obs])]),
        widgets.VBox([widgets.Label('Map zoom:'), w_zoom]),
    out2
])

# VISUALIZATION 3
visual3_container = widgets.VBox([
    widgets.HTML('<h3 style="color:#E9C46A;font-family:Georgia,serif;'
                 'margin:24px 0 4px">3) Species Observation Distributions</h3>'
                 '<p style="color:#8899AA;margin:0 0 8px;font-size:.9em">'
                 'Violin shape = distribution of per-checklist counts. '
                 'Gold dot = median; amber diamond = mean.</p>'),
    widgets.HBox([
        widgets.VBox([widgets.Label('Number of top species to show:'), w_topn]),
        widgets.VBox([widgets.Label('Ordering:'), w_sortby])]),
    out3
])

#### Dashboard

In [14]:
# Create the master layout
dashboard = widgets.VBox([
    visual1_container,
    widgets.HTML('<hr style="border:0; border-top:1px solid #1E3048; margin:20px 0;">'), # Add a separator
    visual2_container,
    widgets.HTML('<hr style="border:0; border-top:1px solid #1E3048; margin:20px 0;">'),
    visual3_container
])

# Display dashboard w/ everything
display(dashboard)

# create the default plots
plot_viz1()
plot_viz2()
plot_viz3()